# 14.3 Q／K 的尺度怎麼影響注意力？

# 第 14 章：逐項理解現代文字模型的設計

前置：第4章基準。這章像替熟悉的車逐一換零件，每節可獨立進入；不要同時換六項再猜哪項有效。結構／數值可先CPU驗證，品質與成本需匹配預算後訓練。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：每張卡片自己調尺度**

沿 feature 軸，不能跨時間偷看別的卡片。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/normalization.svg")))

**先想一想：**K乘100，未正規化與正規化attention哪個更敏感？

點積同時受方向和長度影響。先規範Q/K尺度可以避免某個很長的向量把分數壓倒，另可用可學temperature控制尖銳度。

**把直覺寫成數學：**L2 normalization後點積是cosine；不同實作也可能用RMS或學習scale，需寫清楚。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
q = torch.tensor([[1.0, 1.0]])
k = torch.tensor([[1.0, 0.0], [0.0, 100.0]])
raw = q @ k.T
normalized = F.normalize(q, dim=-1) @ F.normalize(k, dim=-1).T
print(raw, normalized, raw.softmax(-1), normalized.softmax(-1))

**如何讀結果：**這節用L2作直覺；不是宣稱所有QK norm文獻都用同公式。

**只改一件事：**只改scale，固定方向。
